# DrugCLIP Inference Server — Google Colab

Runs the DrugCLIP virtual screening server on a Colab CUDA GPU and exposes it
as a public HTTPS API endpoint via **ngrok** that your local backend calls.

## Before you start
1. **GPU runtime** → Runtime → Change runtime type → **T4 GPU** (or better)
2. **ngrok auth token** → free account at [ngrok.com](https://ngrok.com) → Dashboard → Your Authtoken
3. **Project files** → you will upload `drugclip_inference.py`, `main.py`, and `SMILES.csv`
   from your local `python-worker/` folder when prompted in **Cell 6**

## Run all cells top-to-bottom. Re-run Cell 9 only if the server dies.

In [ ]:
# ── Cell 1: Verify CUDA GPU ────────────────────────────────────────────────
import torch

if not torch.cuda.is_available():
    raise SystemError(
        "No GPU found.  Go to Runtime → Change runtime type → T4 GPU, "
        "then re-run all cells."
    )

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU : {gpu}")
print(f"VRAM: {vram:.1f} GB")
print("CUDA ready")

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────
# Suppress verbose pip output; errors still surface.
import subprocess, sys

pkgs = [
    "fastapi",
    "uvicorn[standard]",
    "pyngrok",
    "pandas",
    "lmdb",
    "scipy",
    "scikit-learn",
    "tqdm",
    "huggingface_hub",
    "rdkit-pypi==2022.9.5",
    "numpy<2",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q"] + pkgs,
    stdout=subprocess.DEVNULL,
)
print("Standard packages installed")

# Uni-Core (unicore) — the backbone framework DrugCLIP uses.
# Must be built from source; requires CUDA headers already present in Colab.
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q",
     "git+https://github.com/dptech-corp/Uni-Core.git"],
    stdout=subprocess.DEVNULL,
)
print("Uni-Core installed")
print("All dependencies ready")

In [ ]:
# ── Cell 3: Clone DrugCLIP and apply required patches ─────────────────────
import os, re

if not os.path.isdir("DrugCLIP"):
    print("Cloning DrugCLIP repo...")
    os.system("git clone -q https://github.com/bowen-gao/DrugCLIP.git")
else:
    print("DrugCLIP already cloned")

# ── Patch 1: tasks/drugclip.py ─────────────────────────────────────────────
# The file does a bare `from IPython import ...` at the top level which crashes
# outside Jupyter unless wrapped in try/except.
tasks_path = "DrugCLIP/unimol/tasks/drugclip.py"
src = open(tasks_path).read()
if "from IPython import embed" in src and "try:" not in src[:80]:
    src = src.replace(
        "from IPython import embed as debug_embedded",
        "try:\n    from IPython import embed as debug_embedded\nexcept ImportError:\n    debug_embedded = lambda: None",
    )
    open(tasks_path, "w").write(src)
    print("Patched tasks/drugclip.py (IPython import)")
else:
    print("tasks/drugclip.py already patched")

# ── Patch 2: models/drugclip.py ────────────────────────────────────────────
# logit_scale parameter is created with device=\"cuda\" hardcoded.
# Replace with a safe runtime check so it works even if CUDA is unavailable.
models_path = "DrugCLIP/unimol/models/drugclip.py"
src = open(models_path).read()
old = 'self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))'
old_cuda = 'self.logit_scale = nn.Parameter(torch.ones([1], device="cuda") * np.log(14))'
new_lines = (
    '_logit_device = "cuda" if torch.cuda.is_available() else "cpu"\n'
    '        self.logit_scale = nn.Parameter(torch.ones([1], device=_logit_device) * np.log(14))'
)
if old_cuda in src:
    src = src.replace(old_cuda, new_lines)
    open(models_path, "w").write(src)
    print("Patched models/drugclip.py (logit_scale device)")
elif "_logit_device" in src:
    print("models/drugclip.py already patched")
else:
    print("WARNING: Could not find logit_scale line to patch — check models/drugclip.py line ~74")

print("DrugCLIP patches applied")

In [ ]:
# ── Cell 4: Download pretrained DrugCLIP weights from HuggingFace (~1.1 GB) ─
import os
from huggingface_hub import hf_hub_download

ckpt_path = "DrugCLIP/weights/pretrain_weights/drugclip.pt"
os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)

if os.path.isfile(ckpt_path):
    size_gb = os.path.getsize(ckpt_path) / 1e9
    print(f"Weights already present ({size_gb:.2f} GB)")
else:
    print("Downloading DrugCLIP weights (1.1 GB) — this takes 1-2 minutes...")
    hf_hub_download(
        repo_id="bgao95/Practical_SBDD",
        filename="pretrain_weights/drugclip.pt",
        local_dir="DrugCLIP/weights",
    )
    print(f"Weights downloaded ({os.path.getsize(ckpt_path)/1e9:.2f} GB)")

In [ ]:
# ── Cell 5: Upload project files ──────────────────────────────────────────
# Upload these three files from your local python-worker/ directory:
#   • drugclip_inference.py
#   • main.py
#   • SMILES.csv
import os
from google.colab import files

required = ["drugclip_inference.py", "main.py", "SMILES.csv"]
missing  = [f for f in required if not os.path.isfile(f)]

if missing:
    print(f"Please upload: {', '.join(missing)}")
    uploaded = files.upload()
    still_missing = [f for f in required if not os.path.isfile(f)]
    if still_missing:
        raise FileNotFoundError(f"Still missing: {still_missing}")
else:
    print("All project files already present")

import pandas as pd
df = pd.read_csv("SMILES.csv")
print(f"SMILES.csv      : {len(df)} molecules")
print(f"drugclip_inference.py : {os.path.getsize('drugclip_inference.py')} bytes")
print(f"main.py         : {os.path.getsize('main.py')} bytes")

In [ ]:
# ── Cell 6: Verify imports and load model onto GPU ─────────────────────────
import sys, os
sys.path.insert(0, "DrugCLIP")

# Suppress verbose unicore startup messages
import warnings
warnings.filterwarnings("ignore")

from drugclip_inference import _load_model, DEVICE
print(f"Inference device : {DEVICE}")

print("Loading model into GPU memory (30-60 s on first run)...")
model, task = _load_model()
print("Model loaded and ready")

import torch
used = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU memory used  : {used:.2f} / {total:.1f} GB")

In [ ]:
# ── Cell 7: Start FastAPI server + expose via ngrok ────────────────────────
#
# 1. Get your free ngrok auth token:
#    https://dashboard.ngrok.com/get-started/your-authtoken
# 2. Paste it below, then run this cell.
# 3. Copy the printed URL into your backend .env as PYTHON_WORKER_URL.
#
# NOTE: The server keeps running as long as this Colab session is alive.
#       If it dies, just re-run this cell.
# ──────────────────────────────────────────────────────────────────────────

NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"  # <── paste your token here

# ──────────────────────────────────────────────────────────────────────────
import subprocess, time, os, sys
from pyngrok import ngrok, conf

if NGROK_AUTH_TOKEN == "YOUR_NGROK_AUTH_TOKEN_HERE":
    raise ValueError(
        "Paste your ngrok auth token into NGROK_AUTH_TOKEN above.\n"
        "Get it at: https://dashboard.ngrok.com/get-started/your-authtoken"
    )

conf.get_default().auth_token = NGROK_AUTH_TOKEN

# Kill any previously running server
os.system("pkill -f 'uvicorn main:app' 2>/dev/null; sleep 1")
ngrok.kill()

# Launch uvicorn in the background
log = open("/tmp/uvicorn.log", "w")
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app",
     "--host", "0.0.0.0", "--port", "8000"],
    stdout=log, stderr=log,
)
time.sleep(5)  # give uvicorn time to start

# Verify server started
if proc.poll() is not None:
    print("Server failed to start. Last 20 lines of log:")
    print(open("/tmp/uvicorn.log").read()[-2000:])
    raise RuntimeError("uvicorn exited — check log above")

# Open public ngrok tunnel
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

# Quick health check
import urllib.request, json as _json
try:
    resp = urllib.request.urlopen(f"{public_url}/health", timeout=10)
    health = _json.loads(resp.read())
    print(f"Health check     : {health}")
except Exception as e:
    print(f"Health check warn: {e} (server may still be starting)")

print()
print("=" * 62)
print(f"  DrugCLIP API LIVE at:")
print(f"  {public_url}")
print("=" * 62)
print()
print("Set this in your backend .env (or docker-compose.yml):")
print(f"  PYTHON_WORKER_URL={public_url}")
print()
print("The server stays alive as long as this Colab session is running.")

In [ ]:
# ── Cell 8 (optional): Quick smoke test ───────────────────────────────────
# Run this to confirm the full inference pipeline works end-to-end before
# connecting your backend.

import urllib.request, json as _json

# 10-atom pocket (alanine dipeptide) + 3 small molecules
payload = {
    "protein_id": "smoke_test",
    "pdb_content": (
        "ATOM      1  N   ALA A   1       1.200   1.100   1.000  1.00  0.00           N\n"
        "ATOM      2  CA  ALA A   1       2.200   1.100   1.000  1.00  0.00           C\n"
        "ATOM      3  C   ALA A   1       2.700   2.200   1.000  1.00  0.00           C\n"
        "ATOM      4  O   ALA A   1       3.700   2.200   1.000  1.00  0.00           O\n"
        "ATOM      5  CB  ALA A   1       2.500   0.000   1.000  1.00  0.00           C\n"
        "ATOM      6  N   GLY A   2       2.000   3.300   1.000  1.00  0.00           N\n"
        "ATOM      7  CA  GLY A   2       2.500   4.400   1.000  1.00  0.00           C\n"
        "ATOM      8  C   GLY A   2       3.700   4.400   2.000  1.00  0.00           C\n"
        "ATOM      9  O   GLY A   2       4.200   5.500   2.000  1.00  0.00           O\n"
        "ATOM     10  N   ALA A   3       4.300   3.300   2.500  1.00  0.00           N\n"
    ),
}

data = _json.dumps(payload).encode()
req  = urllib.request.Request(
    f"{public_url}/discover",
    data=data,
    headers={"Content-Type": "application/json"},
)
print("Sending test request (this can take 1-3 min on first call)...")
with urllib.request.urlopen(req, timeout=600) as resp:
    result = _json.loads(resp.read())

hits = result.get("top_hits", [])
print(f"Top hits returned: {len(hits)}")
for h in hits[:5]:
    print(f"  {h['id']:15s}  score={h['score']:.4f}  smiles={h['smiles'][:30]}")
print("Smoke test PASSED")

## Connecting your local backend

Once Cell 7 prints the ngrok URL, do **one** of the following:

### Option A — `.env` file
Create / edit `backend/.env`:
```
PYTHON_WORKER_URL=https://xxxx-xxx-xxx.ngrok-free.app
PYTHON_WORKER_DRUGCLIP_TIMEOUT_MS=600000
```

### Option B — docker-compose override
```yaml
services:
  backend:
    environment:
      - PYTHON_WORKER_URL=https://xxxx-xxx-xxx.ngrok-free.app
      - PYTHON_WORKER_DRUGCLIP_TIMEOUT_MS=600000
```

Then restart the backend. The `PYTHON_WORKER_DRUGCLIP_TIMEOUT_MS=600000`
(10 min) timeout is already wired into `pythonWorkerService.js` for
`/discover` and `/drugclip` calls.

### Notes
- The ngrok URL **changes every time** you restart Cell 7. Update `.env` accordingly.
- Free ngrok tier allows 1 simultaneous tunnel and 40 req/min — plenty for a hackathon.
- The Colab session disconnects after ~12 h of inactivity. Re-run Cell 7 to reconnect.
- First `/discover` call takes 1-3 min (3D conformer generation + 2× 15-layer transformer encoding for 300 molecules). Subsequent calls with the same model are faster.